Eda Data set Jugadores con ADN Boca

In [1]:
import pandas as pd

# Cargar tus features optimizadas
df = pd.read_csv('adn_boca_features.csv')

# Ver el perfil promedio de cada grupo
perfil_adn = df.groupby('etiqueta')[['rating', 'rendimiento', 'edad', 'partidos', 'participacion_gol']].mean()
print("--- Perfil Promedio por Etiqueta ---")
print(perfil_adn)

# Ver la distribución por posición
print("\n--- Distribución de ADN por Posición ---")
print(pd.crosstab(df['posicion'], df['etiqueta'], normalize='index') * 100)

--- Perfil Promedio por Etiqueta ---
            rating  rendimiento       edad   partidos  participacion_gol
etiqueta                                                                
0         6.687805     2.788736  27.512195  15.743902           0.153179
1         7.443243     4.942153  26.162162  25.135135           0.298141

--- Distribución de ADN por Posición ---
etiqueta            0          1
posicion                        
Attacker    48.437500  51.562500
Defender    37.735849  62.264151
Goalkeeper  18.181818  81.818182
Midfielder  44.615385  55.384615


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Cargar y preparar los datos
df = pd.read_csv('adn_boca_features.csv')

# Guardamos los nombres para los grupos, pero los sacamos de las variables predictoras
grupos = df['nombre']
y = df['etiqueta']

# Eliminamos columnas de tracking/texto que causan overfitting directo
X = df.drop(columns=['nombre', 'temporada', 'etiqueta'])

# Convertimos variables categóricas (como 'posicion') a numéricas
X = pd.get_dummies(X, columns=['posicion'], drop_first=True)

# 2. Configurar GroupKFold (Evita que temporadas del mismo jugador se mezclen en train y test)
gkf = GroupKFold(n_splits=3)

# 3. Configurar modelo con fuerte REGULARIZACIÓN
# Usamos un max_depth bajo (3 o 4) para que aprenda patrones generales y no se memorice los datos
model = RandomForestClassifier(n_estimators=100, max_depth=3, min_samples_leaf=3, random_state=42)

# 4. Loop de Validación Cruzada
print("--- Evaluando Modelo con GroupKFold ---")
fold = 1
for train_idx, test_idx in gkf.split(X, y, grupos):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Entrenar
    model.fit(X_train, y_train)
    
    # Predecir
    preds = model.predict(X_test)
    
    print(f"\nFold {fold} - Accuracy: {accuracy_score(y_test, preds):.2f}")
    # Puedes descomentar la siguiente línea para ver el reporte de cada fold
    # print(classification_report(y_test, preds))
    fold += 1

--- Evaluando Modelo con GroupKFold ---

Fold 1 - Accuracy: 0.91

Fold 2 - Accuracy: 0.92

Fold 3 - Accuracy: 0.95
